# One tree, three interpreters, and no second copy of the model

Ask a modelling team where the units of their model are declared and you get one of two
answers. Either "in the docstring" — which is to say nowhere the computer can check — or
"in the validation module", which is a second copy of the model that drifts from the first
by the third refactor. The parent repo had both, and four separate wrong-number bugs to
show for it.

A model in axiom is a tree of typed `Spec` nodes. A regression is `Add(Mul(Param, Data), ...)`,
a structural model is a `System`, carryover is a `Convolve`, a differential equation is an
`ODESystem`. **One tree, several interpreters:**

| Interpreter | Produces |
|---|---|
| `dimension` | a `Dimension`, or a `DimensionError` naming the node |
| `value` | a numpy array — this *is* `forward()` |
| `latex` | the rendered equation |

The dimension checker is abstract interpretation over the **same tree** the likelihood
evaluates, and the equation in your write-up is typeset from it too. There is no separate
declaration of a model's units that can drift, because there is no separate declaration —
that is rule 3 ("one `forward()`") generalized from one function to one representation.

In [ ]:
from fractions import Fraction

import numpy as np

from axiom.core import (
    Add, Apply, ApplyFn, Const, Convolve, D, Data, DimensionError, Div, Equation, Expr, Gather, Link, LinkFn,
    Model, Mul, ODESystem, Opaque, OpaqueFn, OpaqueRegistry, Param, Pow, Prior, Spec, SupportsForward,
    System, causal_convolve, check, children, data_names, dimension, dimensionless, latex,
    latex_or_unsupported, node_path, params, value, walk,
)

from axiom.display import enable, show_math, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, ORANGE, annotate, caption, compare, curve_band, lines, mark_x

enable();  # every axiom result renders itself from here on

## Leaves: `Const`, `Data`, `Param`

Every leaf declares its dimension. A `Param` with a dimensionless dimension is a **shape**
parameter (it pools across studies); one with a dose or time dimension is a **scale**. That
distinction is not bookkeeping: a shape parameter can be given a prior learned in another
market, and a scale parameter cannot, because the other market's currency is not this one.

In [ ]:
dose = Data(name="dose", dimension=D.currency)
k = Param(name="k", dimension=D.currency, prior=Prior(family="lognormal", hyper={"mu": 3.0, "sigma": 1.0}))
s = Param(name="s", dimension=dimensionless(), prior=Prior(family="gamma", hyper={"alpha": 2.0, "beta": 1.0}))
beta = Param(name="beta", dimension=D.outcome)
one = Const(value=1.0, dimension=dimensionless())

print("k is a scale:", not k.is_shape, "| s is a shape:", s.is_shape)

## A Hill curve

`Pow` with a *variable* exponent needs a dimensionless base — so the dose must be divided by
the half-saturation scale `k` first. That forced `x / k` is the whole scale/shape story: you
cannot raise dollars to a power, and the act of making the base dimensionless is what
introduces the parameter that says *how many dollars count as a lot*.

In [ ]:
x = Div(numerator=dose, denominator=k)
hill: Expr = Mul(factors=(beta, Div(numerator=Pow(base=x, exponent=s),
                                    denominator=Add(terms=(one, Pow(base=x, exponent=s))))))

print(dimension(hill))
show_math(hill)

In [ ]:
grid = np.linspace(0.0, 250.0, 120)
shapes = {f"s = {sv}": value(hill, data={"dose": grid}, params={"k": 50.0, "s": sv, "beta": 10.0})
          for sv in (1.0, 2.0, 4.0)}

fig = lines(
    grid, shapes,
    colors=(BLUE, ORANGE, AQUA),
    title="The two parameters do different jobs, and the dimensions know it",
    subtitle="k = 50 sets where the curve turns; s only sets how sharply — one is in dollars, one is a number",
    x_title="dose (USD)", y_title="outcome",
)
mark_x(fig, 50.0, text="k — the half-saturation scale")
caption(fig, "Every curve passes through half of beta at k, whatever s does. That is why k "
             "carries the dose's dimension and s cannot: only one of them moves when the "
             "currency changes.")

## The same tree, written as mathematics

The constructors above are the canonical form and always work. Every node also carries the
arithmetic operators, so the same tree can be written the way the model is written on paper.
This is sugar with no semantics of its own: the operator form and the longhand form are the
same object, with the same content hash, and they fail the dimension check in the same place.

A bare number lifts to a **dimensionless** `Const`. Nothing local could give it another
dimension, and a literal that quietly borrowed its neighbour's would be exactly the kind of
wrong number the dimension checker exists to catch — so a literal that carries units is still
written out as a `Const`.

In [ ]:
hill_ops = beta * (dose / k) ** s / (1 + (dose / k) ** s)

print("same tree:", hill_ops == hill)
print("same hash:", hill_ops.content_hash() == hill.content_hash())
show_math(hill_ops)

`Add` and `Mul` are n-ary, so `+` and `*` flatten: `a + b + c` is one three-term node, and
associativity is canonicalized — `(a + b) + c` and `a + (b + c)` are the same spec. `sum` works
for the same reason a bare `0` is dropped: zero is the additive identity in every dimension.

`-a` is `(-1) · a` rather than `Apply(fn="neg")`, because `neg` demands a dimensionless argument
and a negated quantity keeps whatever dimension it had. And `alpha[unit_index]` is `Gather` —
a unit-level parameter meeting the panel.

In [ ]:
unit_index = Data(name="unit_index", dimension=dimensionless())
alpha = Param(name="alpha", dimension=D.outcome, shape=(3,))

print("flattened:", len((beta + beta + beta).terms),
      "| associative:", (beta + beta) + beta == beta + (beta + beta))
print("sum:", sum([dose, dose, dose]) == Add(terms=(dose, dose, dose)))
print("negation keeps its dimension:", dimension(-dose))
print("gather:", alpha[unit_index] == Gather(source=alpha, index=unit_index),
      "|", dimension(alpha[unit_index]))

try:
    dimension(dose - 100.0)                       # a bare number is dimensionless
except DimensionError as e:
    print("DimensionError:", e)
print("written out:", dimension(dose - Const(value=100.0, dimension=D.currency)))

In [ ]:
d = np.array([0.0, 25.0, 50.0, 100.0, 200.0])
value(hill, data={"dose": d}, params={"k": 50.0, "s": 2.0, "beta": 10.0})

## The whole posterior, in one call

Parameters can be posterior draws — arrays broadcast by numpy's rules, so one call evaluates
the curve under every draw. This is the mechanism behind every uncertainty band in the
package: the band is not a separate approximation of the curve, it is the same `forward()`
run five hundred times.

In [ ]:
rng = np.random.default_rng(0)
theta = {"k": rng.lognormal(np.log(50), 0.2, size=(500, 1)), "s": rng.gamma(8, 0.25, size=(500, 1)), "beta": 10.0}
curves = value(hill, data={"dose": d[None, :]}, params=theta)
print(curves.shape, "| mean at each dose:", curves.mean(axis=0).round(2))

In [ ]:
draws = value(hill, data={"dose": grid[None, :]}, params=theta)
lo, hi = np.percentile(draws, [5, 95], axis=0)

fig = curve_band(
    grid, draws.mean(axis=0), lo, hi,
    label="posterior mean",
    title="One tree, five hundred draws, one call",
    subtitle="the band is not an approximation of the curve — it is the same forward(), broadcast",
    x_title="dose (USD)", y_title="outcome",
)
annotate(fig, 100.0, float(np.percentile(draws[:, 48], 95)), "5–95% of draws")
caption(fig, "Nothing here knows it is drawing uncertainty. numpy broadcasting over the "
             "draw axis is the entire implementation, which is why the band can never "
             "disagree with the point estimate.")

## The checker names the node

A `log` of a dimensioned quantity, a sum of unlike dimensions, a dimensioned convolution
kernel — each fails at construction-time checking with the path to the offending node. Each
of these is a model somebody wrote, fit, and reported before anyone noticed.

In [ ]:
bad_trees: list[Model] = [
    Apply(fn="log", arg=dose),
    Add(terms=(beta, Mul(factors=(beta, Apply(fn="log", arg=dose))))),
    Pow(base=dose, exponent=s),
    Convolve(signal=dose, kernel=k),
]
refused = []
for t in bad_trees:
    try:
        dimension(t)
    except DimensionError as e:
        refused.append([type(t).__name__, str(e)])
table(refused, headers=("node", "why the checker refuses it"))

A rational *constant* exponent is fine on any base (review A3): the square root of a variance
is a non-integer power of a dimensioned quantity, and a checker that refused it would be
refusing every standard deviation in the package.

In [ ]:
print(dimension(Pow(base=Mul(factors=(dose, dose)), exponent=Fraction(1, 2))))
print(dimension(Pow(base=dose, exponent="1/3")))
show_math(Pow(base=dose, exponent=0.5))

## `check` against a declaration

`check(model, expected)` is what an `Estimand` or a kernel calls to assert the tree derives
to what it claims. A model that claims to produce an outcome and derives to an outcome *per
dollar* is not a modelling preference; it is an error about what the number means.

In [ ]:
print(check(hill, D.outcome))
try:
    check(hill, D.outcome / D.currency)
except DimensionError as e:
    print("refused:", e)

## Transcendentals and links

`Apply` takes an `ApplyFn`; `Link` takes a `LinkFn`. Both require a dimensionless argument
and return a dimensionless value — which is why a log-outcome model needs a reference divide
(`log(y / y_ref)`), and why D9 in the plan exists. `log(revenue)` is not a quantity; it is a
number whose meaning depends on the currency it was computed in.

In [ ]:
fns: list[ApplyFn] = ["exp", "sigmoid", "softplus"]
zero = Const(value=0.0, dimension=dimensionless())
table(
    [[fn, f"{float(value(Apply(fn=fn, arg=zero))):.4f}"] for fn in fns],
    headers=("fn", "f(0)"),
)

link: LinkFn = "log"
y = Data(name="y", dimension=D.outcome)
y_ref = Const(value=100.0, dimension=D.outcome)
print(dimension(Link(fn=link, arg=Div(numerator=y, denominator=y_ref))))

## Carryover as `Convolve`, with an `Opaque` kernel

`Convolve` applies dimensionless weights causally along the last axis. Here the weights come
from an `Opaque` node — user code the tree cannot express. It declares its dimension and is
evaluated through an `OpaqueRegistry`; anything needing introspection (LaTeX) degrades to a
typed `Unsupported` rather than typesetting a lie about what the model does.

In [ ]:
lam = Param(name="lam", dimension=dimensionless())
weights = Opaque(name="geometric", inputs=(lam,), dimension=dimensionless())
carry = Convolve(signal=dose, kernel=weights)

geometric: OpaqueFn = lambda lam, L=6: lam ** np.arange(L)
registry: OpaqueRegistry = {"geometric": geometric}

print(dimension(carry))
impulse_response = value(carry, data={"dose": np.array([100.0, 0, 0, 0, 0, 0])}, params={"lam": 0.6}, opaque=registry)
print(impulse_response)
print(latex_or_unsupported(carry))
print(causal_convolve(np.ones(4), np.array([0.5, 0.25])))

In [ ]:
fig = compare(
    [f"period {i}" for i in range(len(impulse_response))],
    impulse_response,
    highlight="period 0",
    value_fmt="{:.1f}",
    title="What 'causal' means in a causal convolution",
    subtitle="100 units spent in period 0, λ = 0.6 — the response never precedes the spend",
    x_title="outcome",
)
caption(fig, "Every weight sits at or after the impulse. A convolution that let period 0 "
             "borrow from period 1 would be a model that predicts the past, and it is one "
             "sign error away in every hand-rolled implementation.")

## Equations, systems, ODEs

`Equation` requires both sides to share a dimension. `System` checks each equation.
`ODESystem` checks `dim(rhs_i) == dim(state_i) / dim(t)` — the per-period-versus-cumulative
bug class, caught before anything is sampled. In 1.0 the ODE node is dimension-check-only
(review C1).

In [ ]:
response = Equation(lhs=y, rhs=Add(terms=(Param(name="a", dimension=D.outcome), hill)), name="response")
cost = Equation(lhs=Data(name="cost", dimension=D.currency), rhs=Mul(factors=(dose, Const(value=1.0, dimension=dimensionless()))))
model = System(equations=(response, cost))
print(dimension(model))
show_math(model)
print(value(model, data={"dose": d}, params={"k": 50.0, "s": 2.0, "beta": 10.0, "a": 1.0}).shape)

In [ ]:
S = Data(name="S", dimension=D.outcome)
t = Data(name="t", dimension=D.time)
r = Param(name="r", dimension=D.time ** -1)
ode = ODESystem(states=(S,), rhs=(Mul(factors=(r, S)),), time=t)
print(dimension(ode))
show_math(ode)

try:
    dimension(ODESystem(states=(S,), rhs=(S,), time=t))   # forgot the rate: dS/dt = S
except DimensionError as e:
    print("refused:", e)

That last refusal is the picture from `01-dimensions` arriving as a type error: `dS/dt = S`
is the model that decays seven times too fast, and it is now impossible to write down.

## Traversal and serialization

Helpers for walking a tree; and because every node is a `Spec`, a whole model round-trips as
JSON and has a content hash — so the model in a saved analysis is the model, not a
description of one.

In [ ]:
print([p.name for p in params(model)], data_names(model))
print(children(hill)[0], "|", node_path(hill, s))
print([path for path, _ in walk(hill)][:5])
print(Spec.from_json(model.to_json()) == model, model.content_hash()[:16])

## `SupportsForward`

The design layer depends on this protocol: a thing with an `expr` and a `forward` that is the
value interpreter over it. Implementations are not allowed to re-implement the transform
chain — the moment a "fast path" evaluates the curve its own way, the likelihood and the
optimizer are fitting two different models and only one of them is in the write-up.

In [ ]:
class HillSurface:
    expr = hill

    def forward(self, dose, theta):
        return value(self.expr, data=dose, params=theta)

surface = HillSurface()
print(isinstance(surface, SupportsForward), surface.forward({"dose": d}, {"k": 50.0, "s": 2.0, "beta": 10.0}))

## What this bought you

The units, the numbers, and the equation in the paper all come out of one object. A model
that is wrong about its own dimensions cannot be constructed; a model that is right can be
hashed, saved, typeset, evaluated under five hundred draws, and handed to a sampler —
`07-model-spec-and-jax.ipynb` compiles this same tree to jax and checks the two interpreters
agree to eight decimal places, which is gate 9.